# Capítulo 5. Regresión logística

**Aprendizaje y Clasificación Automática con R**  
**Autor:** Jesús Gilberto Rodríguez Escobedo

Este cuaderno es **independiente y autónomo**: puede abrirse directamente sin ejecutar capítulos anteriores.

1. Ejecute primero la celda **Preparación automática y autónoma del capítulo**.
2. Después ejecute las celdas en orden.
3. Si Colab reinicia la sesión, vuelva a ejecutar desde la primera celda.

[Volver al índice de cuadernos Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/00-indice-colabs.ipynb)


In [ ]:
# Preparación automática y autónoma del capítulo
options(repos = c(CRAN = "https://cloud.r-project.org"))

paquetes_libro <- c(
  "ggplot2", "readr", "dplyr", "tidyr", "stringr", "data.table",
  "class", "rpart", "randomForest", "ranger", "e1071", "naivebayes",
  "neuralnet", "cluster", "caret", "factoextra", "scales", "plotly", "DT"
)
faltantes <- paquetes_libro[!vapply(paquetes_libro, requireNamespace, logical(1), quietly = TRUE)]
if (length(faltantes)) install.packages(faltantes)

dir.create("datos/covid19/procesados", showWarnings = FALSE, recursive = TRUE)
dir.create("datos/covid19/muestras", showWarnings = FALSE, recursive = TRUE)
dir.create("datos/covid19/diccionarios", showWarnings = FALSE, recursive = TRUE)

archivos_colab <- c(
  "util_graficas.R" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/util_graficas.R",
  "datos/atus_ml_preparado.csv" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/atus_ml_preparado.csv",
  "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz",
  "datos/covid19/muestras/covid19_mexico_2022_muestra.csv.gz" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/muestras/covid19_mexico_2022_muestra.csv.gz",
  "datos/covid19/diccionarios/diccionario_covid19_ml.csv" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/diccionarios/diccionario_covid19_ml.csv"
)
for (destino in names(archivos_colab)) {
  if (!file.exists(destino)) download.file(archivos_colab[[destino]], destino, mode = "wb", quiet = TRUE)
}
stopifnot(all(file.exists(names(archivos_colab))))
source("util_graficas.R")
cat("Entorno autónomo listo. R:", R.version.string, "\n")


# Regresión logística para clasificación

La formulación matemática de **regresión logística, máxima verosimilitud y clasificación probabilística** se desarrolla con mayor profundidad
en los capítulos 3, 6 y 11 de *Fundamentos Matemáticos del Aprendizaje
Automático* [@rodriguez2026fundamentos].

## Objetivos del capítulo

Al finalizar este capítulo, el lector será capaz de ajustar una regresión logística, predecir probabilidades y evaluar el desempeño.

## Fundamento matemático

La regresión logística utiliza la función:

$$
p = \frac{1}{1 + e^{-z}}
$$

donde $z = \beta_0 + \beta_1x_1 + \cdots + \beta_px_p$.

## Visualización de la función logística


In [ ]:
library(ggplot2)
source("util_graficas.R")

z <- seq(-10, 10, length.out = 200)
datos_logistica <- data.frame(z = z, p = 1 / (1 + exp(-z)))

ggplot(datos_logistica, aes(x = z, y = p)) +
  geom_line(linewidth = 1.2, color = col_azul) +
  labs(
    title = "Función logística",
    subtitle = "Transforma valores reales en probabilidades",
    x = "z",
    y = "Probabilidad"
  ) +
  tema_libro()


## Explicación del código
Se genera una secuencia de valores para $z$ y se calcula su probabilidad usando la función logística.

## Interpretación del resultado
La curva tiene forma de S. Valores negativos producen probabilidades cercanas a cero y valores positivos probabilidades cercanas a uno.

## Cargar paquetes y datos


In [ ]:
library(readr)
library(dplyr)
source("util_graficas.R")

ruta_atus_ml <- "datos/atus_ml_preparado.csv"

if (file.exists(ruta_atus_ml)) {
  atus_ml <- read_csv(ruta_atus_ml, show_col_types = FALSE)
  print("Base preparada cargada correctamente.")
} else {
  atus_ml <- NULL
  print("No se encontró el archivo datos/atus_ml_preparado.csv. Ejecuta primero el capítulo 2.")
}


## Preparar y dividir datos


In [ ]:
if (!is.null(atus_ml)) {
  atus_modelo <- atus_ml |>
    mutate(
      victimas_binaria = ifelse(accidente_con_victimas == "Con víctimas", 1, 0),
      MES = as.factor(MES),
      ID_HORA = as.numeric(ID_HORA),
      DIASEMANA = as.factor(DIASEMANA),
      TIPACCID = as.factor(TIPACCID),
      CAUSAACCI = as.factor(CAUSAACCI),
      accidente_con_victimas = factor(accidente_con_victimas, levels = c("Con víctimas", "Solo daños"))
    )

  set.seed(123)
  idx <- sample(1:nrow(atus_modelo), size = round(0.7 * nrow(atus_modelo)))
  entrenamiento <- atus_modelo[idx, ]
  prueba <- atus_modelo[-idx, ]
}


## Explicación del código
La variable respuesta se transforma a binaria y la base se divide en entrenamiento y prueba.

## Ajustar modelo


In [ ]:
if (exists("entrenamiento")) {
  modelo_logistico <- glm(
    victimas_binaria ~ MES + ID_HORA + DIASEMANA + TIPACCID + CAUSAACCI,
    data = entrenamiento,
    family = binomial
  )

  head(summary(modelo_logistico)$coefficients, 12)
}


## Interpretación del resultado
Los coeficientes estiman el efecto de cada variable sobre el logit de la probabilidad de accidente con víctimas.

## Predicción y matriz de confusión


In [ ]:
if (exists("modelo_logistico")) {
  prueba$probabilidad_victimas <- predict(modelo_logistico, newdata = prueba, type = "response")
  prueba$prediccion_03 <- factor(
    ifelse(prueba$probabilidad_victimas >= 0.3, "Con víctimas", "Solo daños"),
    levels = c("Con víctimas", "Solo daños")
  )

  matriz_confusion_03 <- table(
    Real = prueba$accidente_con_victimas,
    Predicho = prueba$prediccion_03
  )

  matriz_confusion_03
}


## Explicación del código
Las probabilidades se convierten en clases usando un punto de corte de 0.3.

## Métricas


In [ ]:
if (exists("matriz_confusion_03")) {
  VP <- matriz_confusion_03["Con víctimas", "Con víctimas"]
  FN <- matriz_confusion_03["Con víctimas", "Solo daños"]
  FP <- matriz_confusion_03["Solo daños", "Con víctimas"]
  VN <- matriz_confusion_03["Solo daños", "Solo daños"]

  data.frame(
    exactitud = (VP + VN) / (VP + FN + FP + VN),
    sensibilidad = VP / (VP + FN),
    especificidad = VN / (VN + FP)
  )
}


## Interpretación del resultado
La sensibilidad mide la capacidad para detectar accidentes con víctimas; la especificidad mide la capacidad para detectar accidentes de solo daños.

## Distribución de probabilidades


In [ ]:
if (exists("prueba")) {
  ggplot(prueba, aes(x = probabilidad_victimas, fill = accidente_con_victimas)) +
    geom_histogram(bins = 30, alpha = 0.75, position = "identity") +
    escala_clases_fill(name = "Clase real") +
    labs(
      title = "Distribución de probabilidades predichas",
      subtitle = "Regresión logística",
      x = "Probabilidad predicha de accidente con víctimas",
      y = "Frecuencia"
    ) +
    tema_libro()
}


## Materiales complementarios del capítulo
Estos recursos permiten estudiar la regresión logística desde una perspectiva audiovisual, gráfica y práctica.

| Recurso | Utilidad | Abrir o reproducir | Descargar |
|---|---|---|---|
| Video explicativo | Explicación audiovisual de la regresión logística aplicada a problemas de clasificación. | [Ver en YouTube](https://www.youtube.com/watch?v=7CwP40QDeys) | — |
| Presentación en PDF | Diapositivas para lectura, estudio o exposición. | [Ver PDF](recursos/capitulo-05/capitulo-05-regresion-logistica-clasificacion.pdf) | [Descargar PDF](recursos/capitulo-05/capitulo-05-regresion-logistica-clasificacion.pdf){download="capitulo-05-regresion-logistica-clasificacion.pdf"} |
| Presentación editable | Archivo PowerPoint para utilizarlo en clase o adaptarlo. | [Abrir PPTX](recursos/capitulo-05/capitulo-05-regresion-logistica-clasificacion.pptx) | [Descargar PPTX](recursos/capitulo-05/capitulo-05-regresion-logistica-clasificacion.pptx){download="capitulo-05-regresion-logistica-clasificacion.pptx"} |
| Infografía | Síntesis visual de los conceptos, la función logística y el proceso de clasificación. | [Ver infografía](recursos/capitulo-05/capitulo-05-regresion-logistica-clasificacion-infografia.png) | [Descargar PNG](recursos/capitulo-05/capitulo-05-regresion-logistica-clasificacion-infografia.png){download="capitulo-05-regresion-logistica-clasificacion-infografia.png"} |
| Cuaderno Google Colab | Cuaderno autónomo para ejecutar los ejemplos del capítulo sin necesidad de ejecutar los capítulos anteriores. | [Abrir en Google Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/05-regresion-logistica.ipynb) | — |

### Video explicativo

### Vista previa de la infografía

[![Infografía del capítulo 5](recursos/capitulo-05/capitulo-05-regresion-logistica-clasificacion-infografia.png)](recursos/capitulo-05/capitulo-05-regresion-logistica-clasificacion-infografia.png)

**Video del capítulo:** <https://www.youtube.com/watch?v=7CwP40QDeys>

**Curso completo en YouTube:** <https://www.youtube.com/playlist?list=PLDJYd2v7Kt-Q>

La presentación PDF, el archivo PowerPoint y la infografía pueden descargarse desde la versión web del libro.

Este video forma parte de la lista oficial del curso.

[Consultar todos los videos del curso](https://www.youtube.com/playlist?list=PLDJYd2v7Kt-Q)

Los materiales complementarios fueron elaborados con apoyo de **NotebookLM de Google**, a partir del contenido del capítulo, y posteriormente revisados y adaptados por el autor.

## Laboratorio interactivo: regresión logística

Explora cómo el intercepto, la pendiente y el umbral cambian la probabilidad estimada y la clasificación final.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


### Laboratorio disponible en la versión web

El laboratorio permite cambiar los coeficientes, el umbral y el valor de una observación nueva para estudiar la curva logística y la clase predicha.

## Caso aplicado B: regresión logística con COVID-19

Construiremos un modelo para estimar la probabilidad de una **defunción
registrada** a partir de variables demográficas y clínicas.


In [ ]:
library(readr)
library(dplyr)

covid <- read_csv(
  "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz",
  show_col_types = FALSE
)

covid_modelo <- covid |>
  select(
    MURIO,
    EDAD,
    NEUMONIA,
    DIABETES,
    HIPERTENSION,
    OBESIDAD,
    RENAL_CRONICA
  ) |>
  tidyr::drop_na()

set.seed(2026)

indice_entrenamiento <- sample(
  seq_len(nrow(covid_modelo)),
  size = floor(0.80 * nrow(covid_modelo))
)

covid_train <- covid_modelo[indice_entrenamiento, ]
covid_test <- covid_modelo[-indice_entrenamiento, ]

modelo_covid <- glm(
  MURIO ~ EDAD + NEUMONIA + DIABETES +
    HIPERTENSION + OBESIDAD + RENAL_CRONICA,
  data = covid_train,
  family = binomial()
)

summary(modelo_covid)


### Razones de momios


In [ ]:
odds_ratios <- data.frame(
  variable = names(coef(modelo_covid)),
  coeficiente = coef(modelo_covid),
  odds_ratio = exp(coef(modelo_covid))
)

odds_ratios


Un `odds_ratio` mayor que uno indica que, manteniendo constantes las demás
variables, el predictor se asocia con un aumento de los momios del desenlace.
Esto **no demuestra causalidad**.

### Probabilidades para la base de prueba


In [ ]:
covid_test$probabilidad <- predict(
  modelo_covid,
  newdata = covid_test,
  type = "response"
)

head(
  covid_test |>
    arrange(desc(probabilidad)),
  10
)


## Laboratorio interactivo: probabilidad estimada COVID-19

El simulador usa coeficientes estimados previamente con la muestra educativa.
Su propósito es mostrar el funcionamiento matemático de una regresión
logística; **no es una calculadora clínica ni un instrumento de diagnóstico**.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


### Laboratorio disponible en la versión web

El lector puede modificar edad, neumonía y comorbilidades, además del umbral
de clasificación, para observar cómo cambia la probabilidad estimada.

## Conclusión

La regresión logística es un modelo interpretable para clasificación binaria. El punto de corte permite ajustar el balance entre sensibilidad y especificidad.

## Referencias fundamentales de la regresión logística

La formulación moderna de la regresión para respuestas binarias se relaciona con el trabajo de Cox [@cox1958regression]. Para profundizar en ajuste, interpretación, diagnóstico y aplicaciones del modelo logístico puede consultarse a @hosmer2013applied. Una presentación orientada al aprendizaje estadístico y a su implementación en R aparece en @james2021islr.
